# Insight Studio — Colab / Jupyter version
Upload one or many CSV / Excel files, get cleaned data + auto charts + AI insights + PDF/PPT/Excel export.

**Setup:** run cell 1, upload files in cell 2, everything else is automatic.

In [ ]:
# 1. Install dependencies (Colab)
!pip install --quiet pandas openpyxl matplotlib reportlab python-pptx xlsxwriter openai

In [ ]:
# 2. Upload files
try:
    from google.colab import files
    uploaded = files.upload()          # pick one or many .csv / .xlsx
    file_map = {name: content for name, content in uploaded.items()}
except ImportError:
    # local Jupyter — put files in ./data/
    import glob, os
    file_map = {}
    for path in glob.glob('./data/*'):
        with open(path, 'rb') as f:
            file_map[os.path.basename(path)] = f.read()
    print(f'Loaded {len(file_map)} files from ./data/')

In [ ]:
# 3. Parse + clean + summarise
import io, sys, os
sys.path.insert(0, '/app/backend')   # if running inside the app container

# Fallback: inline lightweight versions if backend/analysis.py not available
try:
    from analysis import read_file, clean_dataframe, concat_files, dataset_summary, build_charts, preview_rows, infer_schema
except Exception:
    import pandas as pd
    def read_file(name, content):
        buf = io.BytesIO(content)
        return pd.read_csv(buf) if name.lower().endswith('.csv') else pd.read_excel(buf)
    def clean_dataframe(df): return df.dropna(how='all').dropna(axis=1, how='all')
    def concat_files(frames): return pd.concat(frames, ignore_index=True, sort=False)
    def dataset_summary(df): return {'rows': len(df), 'columns': len(df.columns)}
    def build_charts(df): return []
    def preview_rows(df, n=10): return df.head(n).to_dict(orient='records')
    def infer_schema(df): return [{'name': c, 'type': str(df[c].dtype)} for c in df.columns]

import pandas as pd
frames = [read_file(n, c) for n, c in file_map.items()]
df = clean_dataframe(concat_files(frames))
print(f'{len(df):,} rows · {len(df.columns)} columns')
df.head()

In [ ]:
# 4. Auto charts
import matplotlib.pyplot as plt
charts = build_charts(df)
for c in charts:
    xs = [d['x'] for d in c['data']]
    ys = [d['y'] or 0 for d in c['data']]
    plt.figure(figsize=(8, 3))
    if c['type'] == 'line':
        plt.plot(xs, ys, marker='o')
    else:
        plt.bar(xs, ys)
    plt.title(c['title']); plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# 5. AI insights (optional — needs OpenAI key)
import os, getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key (leave blank to skip AI insights): ')

try:
    from ai_service import generate_insights
except Exception:
    def generate_insights(summary, charts):
        # Lightweight fallback when ai_service isn't available (e.g. outside the app container)
        lines = [f"Dataset has {summary.get('rows', '?')} rows and {summary.get('columns', '?')} columns."]
        if charts:
            lines.append(f"{len(charts)} chart(s) were generated from the data.")
        return lines

if os.environ.get('OPENAI_API_KEY'):
    try:
        insights = generate_insights(dataset_summary(df), charts)
    except Exception as e:
        print(f'AI insights failed, falling back: {e}')
        insights = ['(AI insights unavailable)']
else:
    insights = ['(AI disabled — no API key provided)']

for i, line in enumerate(insights, 1):
    print(f'{i}. {line}')

In [ ]:
# 6. Export PDF / PPT / Excel
summary = dataset_summary(df)

try:
    from exports import build_pdf, build_pptx, build_xlsx
except Exception:
    # Lightweight fallback exporters when the app's exports module isn't available
    from reportlab.lib.pagesizes import letter
    from reportlab.pdfgen import canvas
    from pptx import Presentation
    import xlsxwriter  # noqa: F401 (used via pandas ExcelWriter engine)

    def build_pdf(title, summary, insights, charts, preview):
        buf = io.BytesIO()
        c = canvas.Canvas(buf, pagesize=letter)
        c.setFont('Helvetica-Bold', 16)
        c.drawString(50, 750, title)
        c.setFont('Helvetica', 11)
        y = 720
        c.drawString(50, y, f"Rows: {summary.get('rows', '?')}   Columns: {summary.get('columns', '?')}")
        y -= 30
        for line in insights:
            c.drawString(50, y, f'- {str(line)[:100]}')
            y -= 20
            if y < 50:
                c.showPage(); y = 750
        c.save()
        return buf.getvalue()

    def build_pptx(title, summary, insights, charts):
        prs = Presentation()
        slide = prs.slides.add_slide(prs.slide_layouts[1])
        slide.shapes.title.text = title
        body = slide.placeholders[1].text_frame
        body.text = str(insights[0]) if insights else ''
        for line in insights[1:]:
            body.add_paragraph().text = str(line)
        buf = io.BytesIO()
        prs.save(buf)
        return buf.getvalue()

    def build_xlsx(df, summary, insights):
        buf = io.BytesIO()
        with pd.ExcelWriter(buf, engine='xlsxwriter') as writer:
            df.to_excel(writer, sheet_name='Data', index=False)
        return buf.getvalue()

with open('report.pdf', 'wb') as f: f.write(build_pdf('My data', summary, insights, charts, preview_rows(df)))
with open('deck.pptx', 'wb') as f: f.write(build_pptx('My data', summary, insights, charts))
with open('data.xlsx', 'wb') as f: f.write(build_xlsx(df, summary, insights))
print('Wrote report.pdf, deck.pptx, data.xlsx')